# Imports

In [ ]:
from __future__ import annotations

# File System
from pathlib import Path
import os

# ML & Data
import pandas as pd
import numpy as np
import joblib

# Plot
import matplotlib.pyplot as plt

# sklearn & Models
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score


# Logs & Utils
from functools import wraps
from datetime import datetime
from tqdm import tqdm

# Parameters

In [ ]:
# --- Papermill Parameters ---
# Use a comma-separated string for multiple training sets, e.g., "Bangladesh,Russia_1"
target_campaign = "Ecuador" # Default fallback, should be overriden by papermill
graph_filtering_setting = "all_authors"
folder_output_path = "./results"

# --- Experiment Configurations ---
CAMPAIGNS = ["Egypt_UAE", "Cuba", "Russia", "Venezuela", "Iran", "China"]
FRACTIONS = [0.01, 0.05, 0.10, 0.25, 0.50, 1.0]
MODELS = ["xgboost", "random_forest", "logistic_regression"]


# Utils


In [ ]:
# Robust base directory resolution for cloud/cluster environments
BASE_DIR = Path(os.getenv("WORKSPACE_DIR", "/home/idanmu/Backbone-Graph"))

job_id = os.getenv("SLURM_JOB_ID", datetime.now().strftime("%H%M%S"))
FOLDER_OUTPUT_PATH = BASE_DIR / "results" / target_campaign / datetime.now().strftime("%Y_%m_%d") / job_id
FOLDER_OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
REPORT_PATH = FOLDER_OUTPUT_PATH / "progress_report.txt"

def write_report(msg: str):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"{timestamp} | {msg}"
    print(line)
    with open(REPORT_PATH, "a") as f:
        f.write(line + "\n")

def report_done(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        try:
            result = func(*args, **kwargs)
            write_report(f"Completed: {func.__name__}")
            return result
        except Exception as e:
            write_report(f"FAILED: {func.__name__} | {e}")
            raise
    return wrapper

# --- Model Selection Module ---
def get_classifier(model_name: str, random_state: int = 42, **kwargs):
    model_name = model_name.lower()
    if model_name in ["xgboost", "xgb"]:
        return XGBClassifier(random_state=random_state, eval_metric="logloss", **kwargs)
    if model_name in ["random_forest", "rf"]:
        return RandomForestClassifier(random_state=random_state, **kwargs)
    if model_name in ["logistic_regression", "lr"]:
        return LogisticRegression(random_state=random_state, solver="liblinear", max_iter=1000, **kwargs)
    raise ValueError(f"Unsupported model: {model_name}")

def load_campaign_data(campaign_name: str, prefix_ids: bool = False):

    campaign_name = campaign_name.strip()
    camp_res_dir = BASE_DIR / "results" / f"Labeled_Datasets/{campaign_name}_part_all"
    
    if not camp_res_dir.exists():
        camp_res_dir = BASE_DIR / "results" / f"Labeled_Datasets/{campaign_name}"
        
    dat_dir = BASE_DIR / "data" / f"Labeled_Datasets/{campaign_name}"
    
    feature_files = [f for f in camp_res_dir.rglob(f"*_{campaign_name}_part_all.gzip.parquet") 
                     if f.name.startswith("clustered_") or f.name.startswith("processed_")]

    if not feature_files:
        raise FileNotFoundError(f"PIPELINE ERROR: Feature file not found recursively in: {camp_res_dir}")
        
    df_features = pd.read_parquet(feature_files[0])
    df_labels = pd.read_parquet(dat_dir / f"{campaign_name}_part_all.gzip.parquet")
    # makes sure there are no account ID's duplications across different campaigns
    io_set = set(df_labels[df_labels['is_control'] == False]['accountid'].astype(str))

    if prefix_ids:
        df_features.index = f"{campaign_name}_" + df_features.index.astype(str)
        io_set = {f"{campaign_name}_" + aid for aid in io_set}

    return df_features, io_set

def load_multiple_campaigns(campaigns_csv: str):
    """Allows combining regional datasets seamlessly for model training."""
    campaign_list = campaigns_csv.split(",")
    all_features = []
    combined_io_set = set()
    
    for camp in campaign_list:
        df, io_set = load_campaign_data(camp, prefix_ids=True) 
        all_features.append(df)
        combined_io_set.update(io_set)
        
    return pd.concat(all_features), combined_io_set

# Execution

In [ ]:
# --- Execution Module ---

@report_done
def io_classification(
    node_indicators_df: pd.DataFrame,
    io_authors_set: set,
    model_name: str,
    folder_output_path: str | Path | None = None,
    test_size: float = 0.2,
    random_state: int = 42,
    test_node_indicators_df: pd.DataFrame | None = None,
    test_io_authors_set: set | None = None
):
    """Train and evaluate an IO classifier. Strictly fails on data shape/alignment mismatches."""
    
    if node_indicators_df.empty:
        raise ValueError("FATAL: Training feature set is empty. Aborting to prevent inaccurate results.")

    train_features_df = node_indicators_df.select_dtypes(include=['number'])
    train_cols = train_features_df.columns.tolist()
    X_train = train_features_df.to_numpy()
    
    io_authors_set_str = set(map(str, io_authors_set))
    y_train = node_indicators_df.index.astype(str).isin(io_authors_set_str).astype(int)

    # --- CI/CD SPLIT ROUTING ---
    if test_node_indicators_df is not None and test_io_authors_set is not None:
        write_report("CROSS-CAMPAIGN MODE DETECTED: Bypassing internal train_test_split.")
        
        test_features_df = test_node_indicators_df.select_dtypes(include=['number'])
        test_cols = test_features_df.columns.tolist()
        
        # STRICT ALIGNMENT CHECK: Fail fast if features do not match exactly.
        if train_cols != test_cols:
            mismatched = set(train_cols) ^ set(test_cols)
            raise ValueError(
                f"FATAL ALIGNMENT ERROR: Training and testing features do not match exactly.\n"
                f"Train features count: {len(train_cols)} | Test features count: {len(test_cols)}\n"
                f"Mismatched features: {mismatched}"
            )
            
        X_test = test_features_df.to_numpy()
        test_io_str = set(map(str, test_io_authors_set))
        y_test = test_node_indicators_df.index.astype(str).isin(test_io_str).astype(int)
    else:
        stratify_y = y_train if len(np.unique(y_train)) > 1 else None
        X_train, X_test, y_train, y_test = train_test_split(
            X_train, y_train, test_size=test_size, random_state=random_state, stratify=stratify_y
        )

    write_report(f"Training set: {len(X_train)} samples, Test set: {len(X_test)} samples")

    clf = get_classifier(model_name=model_name, random_state=random_state)
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1] if hasattr(clf, "predict_proba") else y_pred

    report = classification_report(y_test, y_pred, target_names=["Non-IO", "IO"], zero_division=0)
    auc = roc_auc_score(y_test, y_proba) if len(np.unique(y_test)) > 1 else 0.5
    macro_f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

    if folder_output_path:
        folder_output_path = Path(folder_output_path)
        model_artifact_path = folder_output_path / f"{model_name}_model.pkl"
        import joblib
        joblib.dump(clf, model_artifact_path)
        write_report(f"Model artifact saved to {model_artifact_path}")

    return clf, report, auc ,macro_f1

# plot

In [ ]:
def plot_cross_campaign_performance(df_results, metric_col_mean='auc', metric_col_std='auc_std', output_dir=None):
    campaigns = df_results['target_campaign'].unique()
    fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(18, 10))
    axes = axes.flatten()

    colors = {"Graph-Only": "#377eb8", "Combined": "#e41a1c"} # Blue and Red
    markers = {"Graph-Only": "o", "Combined": "^"}

    for idx, camp in enumerate(campaigns):
        ax = axes[idx]
        camp_data = df_results[df_results['target_campaign'] == camp]
        
        for strategy in ["Graph-Only", "Combined"]:
            strat_data = camp_data[camp_data['Strategy'] == strategy].sort_values('train_fraction')
            
            ax.errorbar(
                strat_data['train_fraction'],
                strat_data[metric_col_mean],
                yerr=strat_data[metric_col_std] if metric_col_std and metric_col_std in strat_data.columns else None,
                fmt=f"-{markers[strategy]}",
                color=colors[strategy], label=strategy, capsize=3
            )

        ax.set_title(f"Campaign: {camp}", fontweight='bold')
        ax.set_xlabel('Training Fraction (%)')
        ax.set_ylabel(metric_col_mean.upper())
        ax.legend(fontsize=8)

    plt.tight_layout()
    if output_dir:
        plt.savefig(output_dir / f"strategy_comparison_{metric_col_mean}.png", dpi=300)
    plt.show()

# Hand off

In [ ]:
def run_experiment_sweep(campaigns, fractions, models, graph_setting):
    results_list = []
    
    # Define your Graph-Only columns (Update this list based on your specific indicators)
    GRAPH_COLS = [
        "accountid",
        "follower_count",
        "following_count",
        "is_control",
        "io_author",
        "degree_centrality",
        "harmonic_centrality",
        "betweenness_centrality",
        "eigenvector_centrality",
        "pagerank",
        "core_number",
        "clustering",
        "square_clustering",
        "louvain_community",
        "all_topics_key_score_sum",
        "key_score_dominance",
        "posting_dominance",
        "boost_score",
        "top_topic",
        "hashtags_reuse_12H_accountids",
        "account_mentions_reuse_12H_accountids",
        "ner_entities_reuse_12H_accountids",
        "hashtags_reuse_3D_accountids",
        "account_mentions_reuse_3D_accountids",
        "ner_entities_reuse_3D_accountids"
    ]

    for target_camp in campaigns:
        write_report(f"--- Processing Target Campaign: {target_camp} ---")

        # --- Load target campaign (held-out test set) ---
        try:
            df_target, io_target = load_campaign_data(target_camp, prefix_ids=True)
            y_target = df_target.index.astype(str).isin(set(map(str, io_target))).astype(int)
            df_train_full, df_test, y_train_full, y_test = train_test_split(
                df_target, y_target, test_size=0.2, random_state=42, stratify=y_target
            )
            test_io_set = set(df_test[y_test == 1].index)
        except Exception as e:
            write_report(f"FAILED loading {target_camp}: {e}")
            continue

        # --- Load 5-campaign inter-campaign training set (LOO) ---
        training_camps = [c for c in campaigns if c != target_camp]
        training_csv = ",".join(training_camps)
        try:
            df_inter_train, io_inter = load_multiple_campaigns(training_csv)
            write_report(f"LOO inter-campaign training set loaded: {training_camps} | shape: {df_inter_train.shape}")
        except Exception as e:
            write_report(f"FAILED loading inter-campaign data for {target_camp}: {e}")
            df_inter_train, io_inter = None, None

        for frac in fractions:
            # Intra-campaign fraction
            df_train_frac = df_train_full.sample(frac=frac, random_state=42) if frac < 1.0 else df_train_full
            train_io_frac = set(df_train_frac.index).intersection(io_target)

            # Inter-campaign fraction
            if df_inter_train is not None:
                df_inter_frac = df_inter_train.sample(frac=frac, random_state=42) if frac < 1.0 else df_inter_train
                inter_io_frac = set(df_inter_frac.index).intersection(io_inter)

            for model_name in models:
                for strategy in ["Graph-Only", "Combined"]:
                    available_graph_cols = [c for c in GRAPH_COLS if c in df_train_frac.columns]

                    # ---- INTRA-CAMPAIGN RUN ----
                    try:
                        if strategy == "Graph-Only":
                            train_data = df_train_frac[available_graph_cols]
                            test_data = df_test[available_graph_cols]
                        else:
                            train_data = df_train_frac.select_dtypes(include=['number'])
                            test_data = df_test.select_dtypes(include=['number'])

                        _, report, auc, macro_f1 = io_classification(
                            node_indicators_df=train_data,
                            io_authors_set=train_io_frac,
                            model_name=model_name,
                            test_node_indicators_df=test_data,
                            test_io_authors_set=test_io_set
                        )
                        results_list.append({
                            "training_campaign": target_camp,
                            "target_campaign": target_camp,
                            "graph_filtering_setting": graph_setting,
                            "train_fraction": frac,
                            "Model": model_name,
                            "Strategy": strategy,
                            "experiment_type": "intra",
                            "auc": auc,
                            "macro_f1": macro_f1
                        })
                    except Exception as e:
                        write_report(f"ERROR intra | {target_camp} | {strategy} | {model_name} | {frac}: {e}")

                    # ---- INTER-CAMPAIGN (LOO) RUN ----
                    if df_inter_train is not None:
                        try:
                            if strategy == "Graph-Only":
                                inter_graph_cols = [c for c in GRAPH_COLS if c in df_inter_frac.columns]
                                train_data = df_inter_frac[inter_graph_cols]
                                test_data = df_test[[c for c in inter_graph_cols if c in df_test.columns]]
                            else:
                                train_data = df_inter_frac.select_dtypes(include=['number'])
                                test_data = df_test.select_dtypes(include=['number'])

                            _, report, auc, macro_f1 = io_classification(
                                node_indicators_df=train_data,
                                io_authors_set=inter_io_frac,
                                model_name=model_name,
                                test_node_indicators_df=test_data,
                                test_io_authors_set=test_io_set
                            )
                            results_list.append({
                                "training_campaign": training_csv,
                                "target_campaign": target_camp,
                                "graph_filtering_setting": graph_setting,
                                "train_fraction": frac,
                                "Model": model_name,
                                "Strategy": strategy,
                                "experiment_type": "inter",
                                "auc": auc,
                                "macro_f1": macro_f1
                            })
                        except Exception as e:
                            write_report(f"ERROR inter | {target_camp} | {strategy} | {model_name} | {frac}: {e}")

        del df_target, df_train_full
        if df_inter_train is not None:
            del df_inter_train
        import gc; gc.collect()

    return pd.DataFrame(results_list)


if __name__ == "__main__":
    try:
        # Define the experimental matrix
        # Ensure these names match what is in your folder structure exactly
        target_campaigns = ["Egypt_UAE", "Cuba", "Russia", "Venezuela", "Iran", "China"]
        fractions = [0.01, 0.05, 0.10, 0.25, 0.50, 1.0]
        models_to_run = ["xgboost", "random_forest", "logistic_regression"]
        
        write_report("Starting Full Cross-Campaign Experimental Sweep...")
        
        # 1. Execute the orchestration loop
        results_df = run_experiment_sweep(
            campaigns=target_campaigns, 
            fractions=fractions, 
            models=models_to_run,
            graph_setting=graph_filtering_setting
        )
        
        # 2. Alignment: Add dummy std columns for the plotter if they don't exist
        if 'auc_std' not in results_df.columns:
            results_df['auc_std'] = 0.0
        
        if 'macro_f1_std' not in results_df.columns:
            results_df['macro_f1_std'] = 0.0
        
        # 3. Save the final compiled DataFrame (Requirement 4)
        results_output = Path(folder_output_path) / f"experiment_results_{graph_filtering_setting}.csv"
        results_df.to_csv(results_output, index=False)
        write_report(f"Results table saved to {results_output}")
        
        # 4. Generate the Plots (Requirement 5)
        write_report("Generating performance plots...")
        
        # We override the local campaigns list in the plotter to match target_campaigns
        for metric in ['auc', 'macro_f1']:
            plot_cross_campaign_performance(
                df_results=results_df,
                metric_col_mean=metric,
                metric_col_std=f"{metric}_std" if f"{metric}_std" in results_df.columns else None,
                output_dir=Path(folder_output_path)
            )
        
        write_report("PIPELINE COMPLETED SUCCESSFULLY.")
        
    except Exception as e:
        write_report(f"PIPELINE ABORTED: {e}")